---

# Practical Examination — CUDA Programming

---

**Program:** B.Tech / M.Tech (Computer Science and Engineering)

**Course Outcome:** CO4 — Design and implement parallel algorithms for text and data processing using CUDA

**Difficulty Level:** Moderate

**Dataset / Problem:** DS11

**Problem Statement:** Compute word frequencies in a text corpus using CUDA parallel programming.

---

## 1. Objective

To implement a parallel word frequency computation pipeline using CUDA, where GPU threads perform character-level tokenization and hashing, and atomic operations accumulate per-word counts. The result is verified against a sequential Python-based word counter.

## 2. Theory

### 2.1 Word Frequency Computation

Word frequency (term frequency) is a fundamental NLP operation: given a corpus of text, count the number of occurrences of each distinct word. Sequentially, this is O(N) in the number of characters but becomes a bottleneck for large corpora.

### 2.2 GPU-Accelerated Approach

The strategy is:

1. **Preprocessing (CPU):** Tokenize the corpus into individual words. Convert to lowercase and strip punctuation. Encode each word as a fixed-length integer token (vocabulary index).
2. **Parallel counting (GPU):** Assign one CUDA thread per token. Each thread atomically increments the frequency bucket at index `token_id` in a global frequency array.
3. **Result retrieval (CPU):** Copy the frequency array back from device to host and map indices back to words.

### 2.3 Hashing and Vocabulary Encoding

Since CUDA kernels cannot use Python strings or dictionaries directly, words are mapped to integer indices on the CPU before being transferred to the GPU. This vocabulary index serves as the array offset for the frequency counter.

### 2.4 Atomic Addition

`atomicAdd` is used because multiple threads may attempt to increment the same word's counter simultaneously. Without atomics, this produces incorrect counts due to read-modify-write race conditions.

## 3. Environment Setup

In [1]:
# Verify GPU availability
!nvidia-smi

Mon Jun  1 05:16:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install pycuda --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.2 MB/s eta 0:00:00


## 4. Implementation

In [3]:
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np
import re
import time
from collections import Counter

In [4]:
# -----------------------------------------------------------------------
# CUDA Kernel Definition
# -----------------------------------------------------------------------
# Each thread receives one word token (as an integer vocabulary index).
# It atomically increments the corresponding frequency bucket.
# -----------------------------------------------------------------------

cuda_code = """
__global__ void compute_word_freq(
    int *d_token_ids,
    int *d_freq,
    int  num_tokens
)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_tokens)
    {
        int word_id = d_token_ids[idx];
        // Atomically increment the frequency counter for this word
        atomicAdd(&d_freq[word_id], 1);
    }
}
"""

module = SourceModule(cuda_code)
compute_word_freq = module.get_function("compute_word_freq")

In [5]:
# -----------------------------------------------------------------------
# Sample Text Corpus
# -----------------------------------------------------------------------

corpus = """
To be or not to be that is the question whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune or to take arms against a sea of troubles
and by opposing end them to die to sleep no more and by a sleep to say we end
the heartache and the thousand natural shocks that flesh is heir to tis a consummation
devoutly to be wished to die to sleep to sleep perchance to dream aye there is the rub
for in that sleep of death what dreams may come when we have shuffled off this mortal coil
must give us pause there is the respect that makes calamity of so long life
the parallel computing model allows multiple threads to execute simultaneously
each thread processes one element of the input array the threads cooperate via shared memory
cuda programming enables general purpose computation on graphics processing units
the gpu contains thousands of small cores optimized for parallel workloads
word frequency analysis is a common natural language processing task
the frequency of each word in the corpus reveals important statistical properties
parallel reduction and atomic operations are fundamental cuda programming patterns
the threads per block and grid dimensions must be chosen carefully for performance
memory coalescing is important for achieving peak memory bandwidth on the gpu
"""

print(f"Corpus length: {len(corpus)} characters")

Corpus length: 1311 characters


In [6]:
# -----------------------------------------------------------------------
# CPU Preprocessing: Tokenization and Vocabulary Encoding
# -----------------------------------------------------------------------

# Normalize: lowercase and remove non-alphabetic characters
tokens = re.findall(r'[a-z]+', corpus.lower())

# Build vocabulary: assign each unique word a unique integer index
vocab = {word: idx for idx, word in enumerate(sorted(set(tokens)))}
vocab_size = len(vocab)

# Encode token stream as integer array
token_ids = np.array([vocab[w] for w in tokens], dtype=np.int32)
num_tokens = len(token_ids)

print(f"Total tokens         : {num_tokens}")
print(f"Vocabulary size      : {vocab_size}")
print(f"Sample tokens        : {tokens[:10]}")

Total tokens         : 222
Vocabulary size      : 137
Sample tokens        : ['to', 'be', 'or', 'not', 'to', 'be', 'that', 'is', 'the', 'question']


In [7]:
# -----------------------------------------------------------------------
# GPU Memory Allocation and Data Transfer (Host -> Device)
# -----------------------------------------------------------------------

d_token_ids = cuda.mem_alloc(token_ids.nbytes)
cuda.memcpy_htod(d_token_ids, token_ids)

# Initialize frequency array on GPU to all zeros
h_freq = np.zeros(vocab_size, dtype=np.int32)
d_freq = cuda.mem_alloc(h_freq.nbytes)
cuda.memcpy_htod(d_freq, h_freq)

In [8]:
# -----------------------------------------------------------------------
# Kernel Launch Configuration
# -----------------------------------------------------------------------

THREADS_PER_BLOCK = 256
BLOCKS = (num_tokens + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK

print(f"Threads per block    : {THREADS_PER_BLOCK}")
print(f"Number of blocks     : {BLOCKS}")

Threads per block    : 256
Number of blocks     : 1


In [9]:
# -----------------------------------------------------------------------
# Kernel Execution
# -----------------------------------------------------------------------

start_gpu = time.time()

compute_word_freq(
    d_token_ids,
    d_freq,
    np.int32(num_tokens),
    block=(THREADS_PER_BLOCK, 1, 1),
    grid=(BLOCKS, 1)
)

cuda.Context.synchronize()
gpu_time = time.time() - start_gpu

# Transfer results from Device -> Host
cuda.memcpy_dtoh(h_freq, d_freq)

print(f"GPU Execution Time   : {gpu_time * 1000:.4f} ms")

GPU Execution Time   : 1.3423 ms


## 5. Results and Verification

In [10]:
# Build GPU word-frequency dictionary
idx_to_word = {v: k for k, v in vocab.items()}
gpu_freq = {idx_to_word[i]: int(h_freq[i]) for i in range(vocab_size) if h_freq[i] > 0}

# CPU reference using Python Counter
start_cpu = time.time()
cpu_freq = dict(Counter(tokens))
cpu_time = time.time() - start_cpu

# Correctness check
match = (gpu_freq == cpu_freq)

print("=" * 50)
print("           RESULTS SUMMARY")
print("=" * 50)
print(f"  Total tokens         : {num_tokens}")
print(f"  Unique words         : {vocab_size}")
print(f"  GPU result matches CPU : {match}")
print("-" * 50)
print(f"  GPU time             : {gpu_time * 1000:.4f} ms")
print(f"  CPU time             : {cpu_time * 1000:.4f} ms")
print("=" * 50)

           RESULTS SUMMARY
  Total tokens         : 222
  Unique words         : 137
  GPU result matches CPU : True
--------------------------------------------------
  GPU time             : 1.3423 ms
  CPU time             : 0.1562 ms


The GPU execution time (1.34 ms) exceeded the CPU time (0.16 ms), which is expected at this scale. The corpus is too small to amortize the overhead of kernel launch, host-to-device memory transfers, and atomic contention. For large corpora — millions of tokens with thousands of unique words — the GPU approach scales efficiently since each token is processed independently, making the workload embarrassingly parallel. The benefit of this implementation lies in its scalability, not its performance on a small input.

In [11]:
# Display top-20 most frequent words
top_words = sorted(gpu_freq.items(), key=lambda x: x[1], reverse=True)[:20]

print("\nTop 20 Words by Frequency (GPU Result):")
print("-" * 35)
print(f"  {'Word':<20} {'Count':>5}")
print("-" * 35)
for word, count in top_words:
    print(f"  {word:<20} {count:>5}")
print("-" * 35)


Top 20 Words by Frequency (GPU Result):
-----------------------------------
  Word                 Count
-----------------------------------
  the                     15
  to                      14
  of                       7
  and                      6
  is                       6
  sleep                    5
  a                        4
  be                       4
  for                      4
  that                     4
  in                       3
  memory                   3
  parallel                 3
  threads                  3
  by                       2
  cuda                     2
  die                      2
  each                     2
  end                      2
  frequency                2
-----------------------------------


## 6. Conclusion

A CUDA-based word frequency computation pipeline was implemented and validated. The CPU handled tokenization and vocabulary encoding, converting the text corpus into an integer token array. The GPU kernel then performed parallel frequency counting: each thread processed one token and used `atomicAdd` to safely update the shared frequency array. The GPU output matched the CPU reference exactly. This approach scales efficiently to large corpora because the counting step is fully parallelized across the token stream.

---

*End of Practical — DS11*

---